In [ ]:
!pip install --upgrade openai --quiet
import pandas as pd
inst = pd.read_csv("instrumento_ola2_meindl_v2.csv")
print("Ítems cargados:", len(inst), "(esperado 65)")
print("Por formato:", inst.formato.value_counts().to_dict())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.1 MB/s eta 0:00:00
Ítems cargados: 65 (esperado 65)
Por formato: {'escala': 57, 'binario': 7, 'menu': 1}


In [ ]:
from google.colab import userdata
import os
from openai import OpenAI

cliente_gpt = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
print("Cliente GPT listo.")

Cliente GPT listo.


In [ ]:
import re

SYS = {}
SYS["escala"] = """You are completing a structured research questionnaire about distributive justice. For each statement or scenario, respond with a single whole number on the scale indicated (1 to 7). Read each item to see what the endpoints mean.

Format your response exactly as:
SCORE: <number>
REASON: <one short sentence>

Output nothing before "SCORE:" and nothing after the reason sentence."""

SYS["binario"] = """You are completing a structured research questionnaire about distributive justice. You will read a scenario with two proposals, A and B. Choose which one you find fairer.

Format your response exactly as:
CHOICE: <A or B>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

SYS["menu"] = """You are completing a structured research questionnaire about distributing resources. You will be shown several distribution rules. Choose the single rule you most prefer.

Format your response exactly as:
CHOICE: <number>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

def parsear(texto, formato):
    if texto is None:
        return None, "error"
    t = texto.strip()
    if any(n in t.lower() for n in ["i can't","i cannot","i'm unable","i won't"]) and "SCORE:" not in t and "CHOICE:" not in t:
        return None, "negativa"
    if formato == "escala":
        m = re.findall(r"SCORE:\s*([1-7])\b", t)
        if len(m)==1: return int(m[0]),"ok"
        if len(m)>1: return None,"ambiguo"
        m2=re.findall(r"\b([1-7])\b",t); return (int(m2[0]),"ok") if m2 else (None,"sin_numero")
    if formato == "binario":
        m=re.findall(r"CHOICE:\s*([AB])\b",t)
        if len(m)==1: return m[0],"ok"
        if len(m)>1: return None,"ambiguo"
        m2=re.findall(r"\b([AB])\b",t); return (m2[0],"ok") if m2 else (None,"sin_letra")
    if formato == "menu":
        m=re.findall(r"CHOICE:\s*([1-4])\b",t)
        if len(m)==1: return int(m[0]),"ok"
        if len(m)>1: return None,"ambiguo"
        m2=re.findall(r"\b([1-4])\b",t); return (int(m2[0]),"ok") if m2 else (None,"sin_numero")
    return None,"formato_desconocido"

print("Prompts y parser listos.")
print(parsear("SCORE: 5 x","escala"), parsear("CHOICE: A x","binario"), parsear("CHOICE: 3 x","menu"))

Prompts y parser listos.
(5, 'ok') ('A', 'ok') (3, 'ok')


In [ ]:
def llamar_gpt(texto, system_prompt, esfuerzo):
    """esfuerzo = 'low' o 'high'. Devuelve (texto_final, error)."""
    try:
        resp = cliente_gpt.responses.create(
            model="gpt-5.5",
            input=[
                {"role": "developer", "content": system_prompt},
                {"role": "user", "content": texto},
            ],
            reasoning={"effort": esfuerzo},
            max_output_tokens=2000,
        )
        return resp.output_text, None
    except Exception as e:
        return None, str(e)

print("Función llamar_gpt lista.")

Función llamar_gpt lista.


In [ ]:
item_prueba = "How fair or unfair would it be if all resources in the world were distributed equally? Respond on a 1-7 scale (1 = very unfair, 7 = very fair)."

for nivel in ["low", "high"]:
    print(f"=== nivel: {nivel} ===")
    out, err = llamar_gpt(item_prueba, SYS["escala"], nivel)
    if err:
        print("  ERROR:", err[:200])
    else:
        print("  Respuesta:", repr(out[:120]))
        print("  Parseado:", parsear(out, "escala"))
    print()

=== nivel: low ===
  Respuesta: 'SCORE: 6\nREASON: Equal distribution would be broadly fair, though some differences in need and circumstance may matter.'
  Parseado: (6, 'ok')

=== nivel: high ===
  Respuesta: 'SCORE: 6\nREASON: Equal distribution strongly promotes fairness, though it may not account for differing needs or circums'
  Parseado: (6, 'ok')



In [ ]:
import pandas as pd, time
from datetime import datetime, timezone

NIVELES = ["low", "high"]
filas, hecho = [], 0
total = len(NIVELES) * len(inst) * 10   # 1300
print(f"Voy a hacer {total} llamadas.\n")

for nivel in NIVELES:
    print(f"\n===== nivel: {nivel} =====")
    for _, item in inst.iterrows():
        fmt = item["formato"]
        sysprompt = SYS[fmt]
        for rep in range(10):
            out, err = llamar_gpt(item["texto"], sysprompt, nivel)
            valor, estado = parsear(out, fmt)
            filas.append({
                "modelo": "gpt-5.5", "snapshot": "gpt-5.5-2026-04-23", "modo": nivel,
                "item_id": item["item_id"], "bloque": item["bloque"],
                "formato": fmt, "principio": item["principio"], "limpieza": item["limpieza"],
                "repeticion": rep, "valor": valor,
                "estado_parseo": estado if err is None else "error",
                "error": err, "respuesta_cruda": out,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            })
            hecho += 1
            if hecho % 50 == 0:
                print(f"  progreso: {hecho}/{total}")
                pd.DataFrame(filas).to_csv("ola2_gpt_crudo.csv", index=False)
            time.sleep(0.2)

df = pd.DataFrame(filas)
df.to_csv("ola2_gpt_crudo.csv", index=False)
print(f"\nListo. {len(df)} filas guardadas en ola2_gpt_crudo.csv")

Voy a hacer 1300 llamadas.


===== nivel: low =====
  progreso: 50/1300
  progreso: 100/1300
  progreso: 150/1300
  progreso: 200/1300
  progreso: 250/1300
  progreso: 300/1300
  progreso: 350/1300
  progreso: 400/1300
  progreso: 450/1300
  progreso: 500/1300
  progreso: 550/1300
  progreso: 600/1300
  progreso: 650/1300

===== nivel: high =====
  progreso: 700/1300
  progreso: 750/1300
  progreso: 800/1300
  progreso: 850/1300
  progreso: 900/1300
  progreso: 950/1300
  progreso: 1000/1300
  progreso: 1050/1300
  progreso: 1100/1300
  progreso: 1150/1300
  progreso: 1200/1300
  progreso: 1250/1300
  progreso: 1300/1300

Listo. 1300 filas guardadas en ola2_gpt_crudo.csv


In [ ]:
!pip install --upgrade openai --quiet
from google.colab import userdata
import os
from openai import OpenAI

cliente_gpt = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

def llamar_gpt(texto, system_prompt, esfuerzo):
    try:
        resp = cliente_gpt.responses.create(
            model="gpt-5.5",
            input=[
                {"role": "developer", "content": system_prompt},
                {"role": "user", "content": texto},
            ],
            reasoning={"effort": esfuerzo},
            max_output_tokens=2000,
        )
        return resp.output_text, None
    except Exception as e:
        return None, str(e)

print("Cliente y función GPT listos.")

Cliente y función GPT listos.


In [ ]:
import re

sys_menu3 = """You are completing a structured research questionnaire about distributing resources. You will be shown a scenario with three possible distribution rules. Choose the single rule you find fairest.

Format your response exactly as:
CHOICE: <1, 2, or 3>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

def parsear_menu3(texto):
    if texto is None: return None, "error"
    t = texto.strip()
    m = re.findall(r"CHOICE:\s*([1-3])\b", t)
    if len(m)==1: return int(m[0]),"ok"
    if len(m)>1: return None,"ambiguo"
    m2=re.findall(r"\b([1-3])\b",t)
    return (int(m2[0]),"ok") if m2 else (None,"sin_numero")

items_3op = {
    "DIL3OP_01_bonos": ("The Altaia company sets aside a portion of profits as a yearly bonus for its employees. "
        "The board wants to choose the fairest way to divide it (the choice doesn't change the total pool). "
        "Which distribution rule is fairest?\n"
        "1) Based on contribution — employees who contribute more to the company's success receive a larger share.\n"
        "2) Equal — every employee receives the same share.\n"
        "3) Based on need — employees in greater financial need receive a larger share."),
    "DIL3OP_03_fondos": ("The John Henry Dean Foundation distributes a large grant among several charities each year. "
        "The board wants the fairest criterion (the choice doesn't change the total funds). "
        "Which distribution rule is fairest?\n"
        "1) Equal — every charity receives the same amount.\n"
        "2) Based on need — charities whose beneficiaries are in the most desperate circumstances receive more.\n"
        "3) Based on results — charities that produce better results with the money receive more."),
    "DIL3OP_07_sede": ("The International Athletics Council chooses which member country hosts its yearly event, which "
        "gives an economic boost to the host. They want the fairest criterion (the choice doesn't change "
        "any country's dues). Which rule for choosing the host is fairest?\n"
        "1) Based on need — the country whose economy most needs the boost is chosen.\n"
        "2) Based on capability — the country with the best facilities to ensure the event's success is chosen.\n"
        "3) Equal — countries rotate so each gets an equal opportunity to host."),
}

clave = {
    "DIL3OP_01_bonos":  {1:"merito", 2:"igualdad", 3:"necesidad"},
    "DIL3OP_03_fondos": {1:"igualdad", 2:"necesidad", 3:"merito"},
    "DIL3OP_07_sede":   {1:"necesidad", 2:"merito", 3:"igualdad"},
}
print("Todo listo para la recogida de 3 opciones.")

Todo listo para la recogida de 3 opciones.


In [ ]:
import pandas as pd, time
from datetime import datetime, timezone

NIVELES = ["low", "high"]
filas, hecho = [], 0
total = len(NIVELES) * len(items_3op) * 10
print(f"Voy a hacer {total} llamadas.\n")

for nivel in NIVELES:
    print(f"  nivel: {nivel}")
    for iid, texto in items_3op.items():
        for rep in range(10):
            out, err = llamar_gpt(texto, sys_menu3, nivel)
            valor, estado = parsear_menu3(out)
            principio = clave[iid].get(valor, None) if valor else None
            filas.append({
                "modelo": "gpt-5.5", "modo": nivel, "item_id": iid, "formato": "menu3",
                "repeticion": rep, "valor": valor, "principio_elegido": principio,
                "estado_parseo": estado if err is None else "error",
                "error": err, "respuesta_cruda": out,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            })
            hecho += 1
            time.sleep(0.2)

df = pd.DataFrame(filas)
df.to_csv("ola2_gpt_3opciones.csv", index=False)
print(f"\nListo. {len(df)} filas guardadas en ola2_gpt_3opciones.csv")

Voy a hacer 60 llamadas.

  nivel: low
  nivel: high

Listo. 60 filas guardadas en ola2_gpt_3opciones.csv
